# Differential Equations — Session 16
## Section 4.4: Undetermined Coefficients — Superposition

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Identify when undetermined coefficients applies.
2. Construct a derivative-closed trial space.
3. use superposition for mixed forcing.
4. detect duplication with the complementary function.
5. apply the multiplication rule for resonance.
6. solve an IVP and interpret transient/forced response.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–15 min | Scope and complementary solution |\n| 15–40 min | Trial-form construction |\n| 40–60 min | Coefficient matching and superposition |\n| 60–76 min | Resonance/multiplication rule |\n| 76–88 min | Forced-response simulation |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Method 4.4-A — Undetermined coefficients

For a constant-coefficient linear equation $L[y]=g(x)$, the method applies when $g$ is a finite sum of products of polynomials, exponentials, sines, and cosines.

1. Find $y_c$ from $L[y]=0$.
2. Build a trial $y_p$ from all linearly independent functions generated by differentiating $g$.
3. If a trial term duplicates part of $y_c$, multiply that entire component by the smallest power $x^s$ that removes duplication.
4. Substitute and match coefficients.

### Superposition rule

If $g=g_1+\cdots+g_k$, seek $y_p=y_{p1}+\cdots+y_{pk}$.

### Classroom Checkpoint — Resonant Trial

If the usual undetermined-coefficients trial is already part of the complementary solution, how is it modified?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Trial spaces

| Forcing | Basic trial |
|---|---|
| polynomial degree $n$ | general polynomial degree $n$ |
| $e^{ax}P_n(x)$ | $e^{ax}$ times degree-$n$ polynomial |
| $P_n(x)\cos bx$ or $P_n(x)\sin bx$ | both sine and cosine with degree-$n$ polynomial coefficients |

## 2. Symbolic coefficient matching

Solve

$$
y''-y=3x^2-2x+5.
$$

In [ ]:
x=sp.symbols('x'); A,B,C=sp.symbols('A B C'); yp=A*x**2+B*x+C
expr=sp.Poly(sp.expand(sp.diff(yp,x,2)-yp-(3*x**2-2*x+5)),x)
sol=sp.solve(expr.all_coeffs(),[A,B,C],dict=True); display(sol); display(sp.expand(yp.subs(sol[0])))

## 3. Mixed forcing by superposition

For

$$
y''+y=x+2e^{2x}+3\cos4x,
$$

combine polynomial, exponential, and trigonometric trial components.

In [ ]:
x=sp.symbols('x'); A,B,C,D,E,F=sp.symbols('A B C D E F')
yp=A*x+B+C*sp.exp(2*x)+D*sp.cos(4*x)+E*sp.sin(4*x)
res=sp.expand(sp.diff(yp,x,2)+yp-(x+2*sp.exp(2*x)+3*sp.cos(4*x)))
# Let SymPy solve by expansion into independent terms.
sol=sp.solve(sp.collect(res,[x,sp.exp(2*x),sp.cos(4*x),sp.sin(4*x)],evaluate=False).values(),[A,B,C,D,E],dict=True)
display(sol)

## 4. Duplication and resonance

For

$$
y''+4y=\cos2x,
$$

the usual trial $A\cos2x+B\sin2x$ lies inside $y_c$. Multiply by $x$:

$$
y_p=x(A\cos2x+B\sin2x).
$$

In [ ]:
x=sp.symbols('x')
A,B=sp.symbols('A B')
yp=x*(A*sp.cos(2*x)+B*sp.sin(2*x))
expr=sp.expand_trig(sp.simplify(sp.diff(yp,x,2)+4*yp-sp.cos(2*x)))
display(expr)

# A convenient coefficient choice is A=0, B=1/4.
ytest=x*sp.sin(2*x)/4
display(sp.simplify(sp.diff(ytest,x,2)+4*ytest-sp.cos(2*x)))

## 5. Near resonance versus exact resonance

The undamped oscillator

$$
y''+\omega_0^2y=\cos(\omega t)
$$

has bounded beating when $\omega\ne\omega_0$ and linearly growing amplitude at exact resonance.

In [ ]:
def forced_oscillator(omega0=2.0,omega=1.8,T=40):
    def rhs(t,z): return [z[1],np.cos(omega*t)-omega0**2*z[0]]
    sol=solve_ivp(rhs,(0,T),[0,0],t_eval=np.linspace(0,T,1800),rtol=1e-9,atol=1e-11)
    plt.plot(sol.t,sol.y[0]); plt.xlabel('t'); plt.ylabel('y'); plt.title(f'forcing frequency {omega}, natural frequency {omega0}'); plt.show()
if WIDGETS_AVAILABLE: interact(forced_oscillator,omega0=FloatSlider(min=.5,max=4,step=.1,value=2),omega=FloatSlider(min=.5,max=4,step=.05,value=1.8),T=IntSlider(min=10,max=100,step=5,value=40))
else: forced_oscillator()

## Interactive synthesis — Superposition of forcing components

For the linear equation

$$
y''+y=A_px+A_ee^{-x}+A_t\cos 3x,
$$

the complete forced response is the sum of the responses to the three separate inputs.

In [ ]:
def mixed_forcing_response(Ap=1.0,Ae=1.0,At=1.0,T=15):
    forcing=lambda t:Ap*t+Ae*np.exp(-t)+At*np.cos(3*t)
    sol=solve_ivp(lambda t,z:[z[1],forcing(t)-z[0]],(0,T),[0,0],t_eval=np.linspace(0,T,1200),rtol=1e-9,atol=1e-11)
    plt.plot(sol.t,sol.y[0],label='response')
    plt.plot(sol.t,[forcing(t) for t in sol.t],linestyle='--',label='forcing')
    plt.xlabel('t'); plt.ylabel('value'); plt.title('Superposition of polynomial, exponential, and trigonometric inputs'); plt.legend(); plt.show()
if WIDGETS_AVAILABLE:
    interact(mixed_forcing_response,Ap=FloatSlider(min=-2,max=2,step=.25,value=1),Ae=FloatSlider(min=-2,max=2,step=.25,value=1),At=FloatSlider(min=-2,max=2,step=.25,value=1),T=IntSlider(min=5,max=30,step=5,value=15))
else: mixed_forcing_response()

## Classroom Checkpoint — Exit Check

Choose a trial for

$$
y''-2y'+5y=x^2e^x\cos2x.
$$

> Pause here. Let students commit to an answer before running the next cell.